In [1]:
import pandas as pd
import numpy as np
import ast
import re
import matplotlib.pyplot as plt

import requests
import time



In [ ]:
# 1. LOAD DRUGS DIRECTLY FROM FILE
df_drugs = pd.read_csv("C:/Katieryb/Pipelines/proteins/drug_list.csv") 

#extract the list of drug names from file's "Drug" column
drug_list = df_drugs['Drug'].tolist()

# 2. PASTE OPENFDA API KEY HERE
API_KEY = "KJfMeoPhO5nL1XiINX3uxCj4J9xjkNsyb22spbXt"

all_mild_reports = []

for drug in drug_list:
    print(f"Fetching non-extreme feedback for: {drug}...")
    
    # 3. EXCLUDE EXTREME CASES (We explicitly query serious:2 for non-extreme)
    url = (
        f'https://api.fda.gov/drug/event.json'
        f'?api_key={API_KEY}'
        f'&search=patient.drug.medicinalproduct:"{drug}" AND serious:2'
        f'&limit=50'
    )
    
    response = requests.get(url)
    
    if response.status_code == 200:
        results = response.json().get('results', [])
        for record in results:
            # Grabbing the mild/common side effects listed in the record
            reactions = [r.get('reactionmeddrapt') for r in record.get('patient', {}).get('reaction', [])]
            
            all_mild_reports.append({
                'Drug': drug,
                'Report_ID': record.get('safetyreportid'),
                'Mild_Side_Effects': reactions
            })
    elif response.status_code == 404:
        print(f"No non-extreme records found for {drug}.")
        
    # Brief pause to keep the connection smooth and comply with API limits
    time.sleep(0.2)

#convert all fetched bioinformatics data into a clean dataframe table
df_results = pd.DataFrame(all_mild_reports)
print("\nDownload complete! Here is a sample of the non-extreme data: ---")
print(df_results.head())


# Save the results table directly to folder as a new CSV spreadsheet
df_results.to_csv("C:/Katieryb/Pipelines/proteins/mild_effects.csv", index=False)

Fetching non-extreme feedback for: Ibuprofen...
Fetching non-extreme feedback for: Acetaminophen...
Fetching non-extreme feedback for: Aspirin...
Fetching non-extreme feedback for: Naproxen...
Fetching non-extreme feedback for: Diphenhydramine...
Fetching non-extreme feedback for: Loratadine...
Fetching non-extreme feedback for: Cetirizine...
Fetching non-extreme feedback for: Amoxicillin...
Fetching non-extreme feedback for: Azithromycin...
Fetching non-extreme feedback for: Lisinopril...
Fetching non-extreme feedback for: Metoprolol...
Fetching non-extreme feedback for: Amlodipine...
Fetching non-extreme feedback for: Metformin...
Fetching non-extreme feedback for: Fluoxetine...
Fetching non-extreme feedback for: Sertraline...
Fetching non-extreme feedback for: Clozapine...
Fetching non-extreme feedback for: Insulin...

--- Download complete! Here is a sample of your non-extreme data: ---
        Drug Report_ID                                  Mild_Side_Effects
0  Ibuprofen  10004183

In [3]:
faers_file = "C:/Katieryb/Pipelines/proteins/mild_effects.csv"

faers = pd.read_csv(faers_file)

print("Shape:", faers.shape)
print("\nColumns:")
print(faers.columns.tolist())

print("\nFirst 5 rows:")
display(faers.head())


print(type(faers.loc[0, "Mild_Side_Effects"]))
print(faers.loc[0, "Mild_Side_Effects"])



Shape: (850, 3)

Columns:
['Drug', 'Report_ID', 'Mild_Side_Effects']

First 5 rows:


,Drug,Report_ID,Mild_Side_Effects
0,Ibuprofen,10004183,['Weight loss poor']
1,Ibuprofen,10004379,"['Malaise', 'Pain', 'Feeling abnormal']"
2,Ibuprofen,10004622,"['White blood cell count increased', 'Platelet..."
3,Ibuprofen,10004874,['Drug hypersensitivity']
4,Ibuprofen,10005416,"['Adverse event', 'Influenza like illness']"


<class 'str'>
['Weight loss poor']


In [4]:

#Convert the side-effect lists into individual events
def parse_side_effects(value):
    """
    Convert the Mild_Side_Effects column into a clean Python list.
    """
    
    if pd.isna(value):
        return []
    
    # Already a list
    if isinstance(value, list):
        return value
    
    # String representation of a list
    if isinstance(value, str):
        value = value.strip()
        
        try:
            parsed = ast.literal_eval(value)
            
            if isinstance(parsed, list):
                return parsed
            
            return [str(parsed)]
        
        except (ValueError, SyntaxError):
            # Fallback if the string isn't valid Python-list syntax
            return [value]
    
    return []

In [5]:

faers["Side_Effects"] = faers["Mild_Side_Effects"].apply(
    parse_side_effects
)

print(faers[["Drug", "Report_ID", "Side_Effects"]].head())



        Drug  Report_ID                                       Side_Effects
0  Ibuprofen   10004183                                 [Weight loss poor]
1  Ibuprofen   10004379                  [Malaise, Pain, Feeling abnormal]
2  Ibuprofen   10004622  [White blood cell count increased, Platelet co...
3  Ibuprofen   10004874                            [Drug hypersensitivity]
4  Ibuprofen   10005416            [Adverse event, Influenza like illness]


In [6]:
#explode doc
faers_exploded = faers.explode(
    "Side_Effects"
).copy()

print("Rows after exploding:", len(faers_exploded))

display(
    faers_exploded[
        ["Drug", "Report_ID", "Side_Effects"]
    ].head(20)
)

Rows after exploding: 2468


,Drug,Report_ID,Side_Effects
0,Ibuprofen,10004183,Weight loss poor
1,Ibuprofen,10004379,Malaise
1,Ibuprofen,10004379,Pain
1,Ibuprofen,10004379,Feeling abnormal
2,Ibuprofen,10004622,White blood cell count increased
2,Ibuprofen,10004622,Platelet count decreased
2,Ibuprofen,10004622,Contusion
3,Ibuprofen,10004874,Drug hypersensitivity
4,Ibuprofen,10005416,Adverse event
4,Ibuprofen,10005416,Influenza like illness


In [7]:
#clean up

faers_exploded["Adverse_Event"] = (
    faers_exploded["Side_Effects"]
    .astype(str)
    .str.strip()
)

#remove empty/null values

faers_exploded = faers_exploded[
    faers_exploded["Adverse_Event"].notna()
]

faers_exploded = faers_exploded[
    faers_exploded["Adverse_Event"] != ""
]

faers_exploded = faers_exploded[
    faers_exploded["Adverse_Event"].str.lower() != "nan"
]

In [ ]:
#check how many unique events have

print(
    "Unique adverse events:",
    faers_exploded["Adverse_Event"].nunique()
)

print(
    "Unique drugs:",
    faers_exploded["Drug"].nunique()
)

print(
    "Unique reports:",
    faers_exploded["Report_ID"].nunique()
)


print("\nEvents with highest number of reports:")

event_counts = (
    faers_exploded
    .groupby("Adverse_Event")["Report_ID"]
    .nunique()
    .sort_values(ascending=False)
)

print(event_counts.head(30))

Unique adverse events: 547
Unique drugs: 17
Unique reports: 750

Events with highest number of reports:
Adverse_Event
Drug ineffective            82
Fatigue                     50
Headache                    49
Diarrhoea                   48
Drug hypersensitivity       46
Nausea                      41
Dizziness                   35
Pain                        31
Dyspnoea                    30
Rash                        28
Product quality issue       26
Abdominal pain upper        23
Insomnia                    21
Asthenia                    20
Vomiting                    19
Anaemia                     18
Weight decreased            18
Pain in extremity           17
Somnolence                  17
Off label use               17
Arthralgia                  17
Platelet count decreased    16
Feeling abnormal            16
Malaise                     15
Nasopharyngitis             15
Abdominal pain              15
Extra dose administered     15
Haemoglobin decreased       15
Abdominal dist

In [9]:
drug_event_counts = (
    faers_exploded
    .groupby(["Drug", "Adverse_Event"])["Report_ID"]
    .nunique()
    .reset_index(name="Report_Count")
)

print(drug_event_counts.shape)

display(
    drug_event_counts.head(20)
)


drug_event_counts.to_csv(
    "C:/Katieryb/Pipelines/proteins/drug_adverse_event_counts.csv",
    index=False
)

print("Saved drug_adverse_event_counts.csv")

(1621, 3)


,Drug,Adverse_Event,Report_Count
0,Acetaminophen,Abasia,1
1,Acetaminophen,Abdominal distension,2
2,Acetaminophen,Abdominal pain,1
3,Acetaminophen,Abdominal pain lower,1
4,Acetaminophen,Abdominal pain upper,4
5,Acetaminophen,Adverse event,1
6,Acetaminophen,Anaemia,2
7,Acetaminophen,Anxiety,2
8,Acetaminophen,Application site rash,1
9,Acetaminophen,Arthralgia,4


Saved drug_adverse_event_counts.csv
